In [1]:
from pathlib import Path
import re
import unicodedata

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

DATA_FOLDER = Path("../../data")
FEATURE_FOLDER = DATA_FOLDER / "features" / "tfidf"

MATRIX_PATH = FEATURE_FOLDER / "tfidf_matrix.npz"
VECTORIZER_PATH = FEATURE_FOLDER / "tfidf_vectorizer.joblib"
DOCUMENTS_PATH = FEATURE_FOLDER / "tfidf_documents.csv"

print(f"Feature folder: {FEATURE_FOLDER.resolve()}")

Feature folder: C:\Works\uts-ir\src\data\features\tfidf


In [2]:
X_main = load_npz(MATRIX_PATH)
vectorizer = joblib.load(VECTORIZER_PATH)
df = pd.read_csv(DOCUMENTS_PATH, low_memory=False)

df.head()

,id,title,abstract,title_translated,abstract_translated,processed_token_title,processed_token_abstract,processed_title,processed_abstract,processed_text
0,2541-3279-1,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,"['hubung', 'promosi', 'pustaka', 'lalu', 'maja...","['teliti', 'dilatarbelakangi', 'kembang', 'jam...",hubung promosi pustaka lalu majalah dinding mi...,teliti dilatarbelakangi kembang jaman makin ma...,hubung promosi pustaka lalu majalah dinding mi...
1,2541-3279-2,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,"['hubung', 'guna', 'search', 'features', 'jour...","['teliti', 'bahas', 'hubung', 'guna', 'search'...",hubung guna search features journals penuh but...,teliti bahas hubung guna search features journ...,hubung guna search features journals penuh but...
2,2541-3279-3,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,"['hubung', 'sedia', 'fasilitas', 'pustaka', 'm...","['teliti', 'bahas', 'hubung', 'sedia', 'fasili...",hubung sedia fasilitas pustaka minat kunjung s...,teliti bahas hubung sedia fasilitas pustaka mi...,hubung sedia fasilitas pustaka minat kunjung s...
3,2541-3279-4,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,"['pengaruh', 'kerja', 'layan', 'prima', 'cente...","['masalah', 'jadi', 'kaji', 'teliti', 'kena', ...",pengaruh kerja layan prima center of informati...,masalah jadi kaji teliti kena layan prima inti...,pengaruh kerja layan prima center of informati...
4,2541-3279-5,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,"['kontribusi', 'literasi', 'informasi', 'mahas...","['fenomena', 'pesat', 'kembang', 'ilmu', 'tahu...",kontribusi literasi informasi mahasiswa proses...,fenomena pesat kembang ilmu tahu teknologi inf...,kontribusi literasi informasi mahasiswa proses...


In [15]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.casefold()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b", " ", text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

stopword_factory = StopWordRemoverFactory()
stopwords_id = set(stopword_factory.get_stop_words())
stemmer = StemmerFactory().create_stemmer()


def preprocess_query(query):
    normalized_query = normalize_text(query)
    tokens = normalized_query.split()
    tokens_no_stopwords = [
        token for token in tokens
        if token not in stopwords_id and len(token) > 1
    ]
    tokens_stemmed = [stemmer.stem(token) for token in tokens_no_stopwords]
    processed_query = " ".join(tokens_stemmed)

    return {
        "normalized_query": normalized_query,
        "tokens": tokens,
        "tokens_no_stopwords": tokens_no_stopwords,
        "tokens_stemmed": tokens_stemmed,
        "processed_query": processed_query,
    }

In [ ]:
def search_articles(query, top_k=10):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query tidak boleh kosong. Masukkan kata atau frasa pencarian.")
    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k harus berupa bilangan bulat positif.")

    query_steps = preprocess_query(query)
    processed_query = query_steps["processed_query"]

    if not processed_query:
        print("Query menjadi kosong setelah preprocessing. Coba gunakan istilah yang lebih spesifik.")
        return pd.DataFrame()

    query_vector = vectorizer.transform([processed_query])
    query_terms = processed_query.split()
    out_of_vocabulary = [term for term in query_terms if term not in vectorizer.vocabulary_]

    if query_vector.nnz == 0:
        print("Tidak ada term query yang ditemukan dalam vocabulary TF-IDF.")
        if out_of_vocabulary:
            print("Term di luar vocabulary:", ", ".join(dict.fromkeys(out_of_vocabulary)))
        return pd.DataFrame()

    scores = cosine_similarity(query_vector, X_main).ravel()
    ranked_indices = np.argsort(scores)[::-1]
    ranked_indices = [idx for idx in ranked_indices if scores[idx] > 0][:top_k]

    if not ranked_indices:
        print("Tidak ditemukan dokumen dengan skor cosine similarity di atas nol.")
        return pd.DataFrame()

    columns_available = set(df.columns)
    display_columns = [
        column for column in ["id", "title", "abstract"]
        if column in columns_available
    ]
    results = df.iloc[ranked_indices][display_columns].copy()
    results.insert(0, "rank", np.arange(1, len(results) + 1))
    results["cosine_similarity"] = scores[ranked_indices]
    results = results.sort_values("cosine_similarity", ascending=False).reset_index(drop=True)

    print(f"Query            : {query}")
    print(f"Term ditemukan   : {query_vector.nnz} term(s) dalam vocabulary.")
    if out_of_vocabulary:
        print("Term di luar vocabulary:", ", ".join(dict.fromkeys(out_of_vocabulary)))

    return results

In [17]:
query = "literasi informasi mahasiswa"
results = search_articles(query, top_k=10)

display(results)

Query            : literasi informasi mahasiswa
Term ditemukan   : 3 fitur nonzero


,rank,id,title,abstract,cosine_similarity
0,1,2797-9393-24,peran pustakawan referensi perpustakaan uin im...,perpustakaan perguruan tinggi memiliki peran p...,0.595662
1,2,2598-7852-71,urgensi kelas literasi informasi bagi mahasisw...,artikel ini bertujuan untuk mengetahui urgensi...,0.594022
2,3,2657-2346-104,ngkat literasi informasi dan literasi akademik...,mahasiswa memiliki peran tidak hanya sebagai p...,0.562624
3,4,2685-8363-25,strategi peningkatan kompetensi literasi infor...,artikel ini membahas mengenai strategi meningk...,0.546563
4,5,2541-3279-5,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,0.542460
5,6,2829-9531-18,evaluasi tingkat literasi informasi mahasiswa ...,penelitian ini bertujuan untuk mengidentifikas...,0.540844
6,7,2962-5599-5,persepsi kemampuan literasi informasi di dalam...,perpustakaan sebagai sebuah pusat belajar memi...,0.534898
7,8,2540-9239-182,identifying information literacy skills of new...,perpustakaan sekolah pascasarjana uin syarif h...,0.517934
8,9,2598-3040-94,pengalaman mahasiswa menggunakan perpustakaan ...,mahasiswa memiliki kebutuhan informasi yang ha...,0.515106
9,10,2723-7605-40,kemampuan literasi informasi mahasiswa prodi m...,penelitian ini bertujuan menganalisis kemampua...,0.513060


In [18]:
test_queries = [
    "literasi informasi mahasiswa",
    "promosi perpustakaan",
    "sistem informasi perpustakaan",
]

for test_query in test_queries:
    test_results = search_articles(test_query, top_k=5)
    display(test_results[[
        column for column in ["rank", "id", "title", "cosine_similarity"]
        if column in test_results.columns
    ]])

Query            : literasi informasi mahasiswa
Term ditemukan   : 3 fitur nonzero


,rank,id,title,cosine_similarity
0,1,2797-9393-24,peran pustakawan referensi perpustakaan uin im...,0.595662
1,2,2598-7852-71,urgensi kelas literasi informasi bagi mahasisw...,0.594022
2,3,2657-2346-104,ngkat literasi informasi dan literasi akademik...,0.562624
3,4,2685-8363-25,strategi peningkatan kompetensi literasi infor...,0.546563
4,5,2541-3279-5,kontribusi literasi informasi mahasiswa terhad...,0.542460


Query            : promosi perpustakaan
Term ditemukan   : 1 fitur nonzero
Term di luar vocabulary: perpustakaan


,rank,id,title,cosine_similarity
0,1,2502-4108-87,optimalisasi web untuk promosi perpustakaan,0.629627
1,2,2723-6234-24,library promotion by governance institute of h...,0.595943
2,3,2723-6234-15,analysis of college library promotion activiti...,0.589473
3,4,2715-274x-63,kegiatan mempromosikan perpustakaan: sebuah pe...,0.548605
4,5,2685-2187-28,strategi promosi perpustakaan khusus: studi pa...,0.511493


Query            : sistem informasi perpustakaan
Term ditemukan   : 2 fitur nonzero
Term di luar vocabulary: perpustakaan


,rank,id,title,cosine_similarity
0,1,2598-3040-173,pengembangan sistem informasi perpustakaan unt...,0.557586
1,2,2477-0361-142,analisis dan perancangan desain sistem informa...,0.488163
2,3,2723-2778-6,pengembangan sistem informasi karya ilmiah ber...,0.467045
3,4,2598-3040-47,daur hidup pengembangan sistem untuk otomasi p...,0.437339
4,5,2723-0953-11,"planning and implementation of the ""library of...",0.434951
